### EDA

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
X = pd.read_json('data/features.json',orient='records')
y = pd.read_json('data/target.json',orient='records')

print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')

In [ ]:
# Check class distribution
print('Target Distribution:')
print(y.value_counts())
print('\nPercentages:')
print(y.value_counts(normalize=True) * 100)

# Visualize
plt.figure(figsize=(6, 5))
y['label'].value_counts().sort_index().plot(kind='bar', color=['steelblue', 'salmon'])
plt.title('Class Distribution (0 = Benign, 1 = Pathogenic)')
plt.xlabel('Label')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('figures/class_distribution.png')
plt.show()

In [ ]:
print('Top 20 genes by variant count:')
top_20 = X['GeneSymbol'].value_counts().head(20)
print(top_20)

plt.figure(figsize=(12, 6))
top_20.plot(kind='bar', color='steelblue')
plt.title('Top 20 genes by variant count')
plt.xlabel('GeneSymbol')
plt.ylabel('Number of Variants')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
df_combined = X.copy()
df_combined['label'] = y['label']

top_20_genes = top_20.index

top_genes_df = df_combined[df_combined['GeneSymbol'].isin(top_20_genes)]
gene_pathogenic_rate = top_genes_df.groupby('GeneSymbol').apply(
    lambda x: (x['label'] == 1).sum() / len(x) * 100
).sort_values(ascending=False)

print('Pathogenic rate for top 20 genes:')
print(gene_pathogenic_rate)

plt.figure(figsize=(12, 6))
gene_pathogenic_rate.plot(kind='bar', color='steelblue')
plt.title('Pathogenic rate for top 20 genes')
plt.xlabel('GeneSymbol')
plt.ylabel('Pathogenic Rate (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('figures/top_genes.png')
plt.show()

In [ ]:
# Pathogenic rate by chromosome
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y']
chrom_pathogenic_rate = df_combined.groupby('Chromosome').apply(
    lambda x: (x['label'] == 1).sum() / len(x) * 100
).reindex(chrom_order).dropna()

plt.figure(figsize=(14, 5))
chrom_pathogenic_rate.plot(kind='bar', color='steelblue')
plt.title('Pathogenic rate by chromosome')
plt.xlabel('Chromosome')
plt.ylabel('Pathogenic Rate (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('figures/pathogenic_rate_by_chromosome.png')
plt.show()

In [ ]:
# Nucleotide substitution analysis
df_combined['substitution'] = df_combined['ReferenceAlleleVCF'] + '→' + df_combined['AlternateAlleleVCF']

substitution_counts = df_combined['substitution'].value_counts()
sub_pathogenic_rate = df_combined.groupby('substitution').apply(
    lambda x: (x['label'] == 1).sum() / len(x) * 100
).reindex(substitution_counts.index)

print('Substitution counts:')
print(substitution_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

substitution_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Substitution type counts')
axes[0].set_xlabel('Substitution')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

sub_pathogenic_rate.plot(kind='bar', ax=axes[1], color='salmon')
axes[1].set_title('Pathogenic rate by substitution type')
axes[1].set_xlabel('Substitution')
axes[1].set_ylabel('Pathogenic Rate (%)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('figures/substitution_analysis.png')
plt.show()